# PDF Analysis and ML Parsing Model

This notebook demonstrates how to extract text from image-based PDFs using OCR and then build a small Machine Learning model to parse the structured data (Semester, Subjects, Grades).

## prerequisites
1. **Tesseract OCR**: You must have Tesseract installed on your system.
   - **Windows**: Download installer from [UB-Mannheim/tesseract](https://github.com/UB-Mannheim/tesseract/wiki)
   - Add Tesseract to your System PATH.
2. **Poppler**: Required for `pdf2image`.
   - **Windows**: Download binary, extract, and add `bin/` to PATH.

## Setup

In [ ]:
!pip install pytesseract pdf2image scikit-learn pandas numpy opencv-python

In [1]:
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import numpy as np
import re
import os

# Set Tesseract Path (Update this if Tesseract is not in PATH)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

ModuleNotFoundError: No module named 'pytesseract'

## 1. OCR Extraction
Since the PDF contains images, standard text extraction fails. We use `pdf2image` to convert pages to images and `pytesseract` to read text.

In [ ]:
pdf_path = 'Semester 4.pdf'

full_text = ""

if os.path.exists(pdf_path):
    try:
        # Convert PDF to images
        print("Converting PDF to images...")
        images = convert_from_path(pdf_path)
        
        print(f"Extracted {len(images)} pages. Running OCR...")
        for i, image in enumerate(images):
            text = pytesseract.image_to_string(image)
            full_text += text + "\n"
            print(f"-- Page {i+1} Processed --")
            
    except Exception as e:
        print(f"Error: {e}")
        print("Ensure Tesseract and Poppler are installed and in PATH.")
else:
    print("PDF file not found.")

## 2. Synthetic Data (Fallback)
If OCR failed above (due to missing dependencies), we use this synthetic text that mimics the PDF structure to demonstrate the ML model.

In [ ]:
if not full_text.strip():
    print("Using synthetic data for demonstration...")
    # We updated the synthetic data to better match the visual table format
    # Note: Often a Grade Point (integer) appears before the Letter Grade.
    full_text = """
    ANNA UNIVERSITY :: CHENNAI - 600 025.
    OFFICE OF THE CONTROLLER OF EXAMINATIONS
    Provisional Results of April / May Examination,2024.
    Semester-04
    Semester-4 21MA401 - PROBABILITY AND STATISTICS 7 B+ 4 Pass
    Semester-4 21CS401 - DATABASE MANAGEMENT SYSTEMS 8 A 3 Pass
    Semester-4 21CS402 - DESIGN AND ANALYSIS OF ALGORITHMS 6 B 3 pass
    Semester-4 21CS403 - OPERATING SYSTEMS 9 A+ 3 Pass
    Semester-4 21CS411 - DATABASE MANAGEMENT SYSTEMS LABORATORY 10 O 2 Pass
    Semester-4 21HS401 - ENVIRONMENTAL SCIENCE AND ENGINEERING 5 C 0 Pass
    Semester-05
    Semester-5 21CS501 - COMPUTER NETWORKS 0 RA 0 Fail
    """

lines = [line.strip() for line in full_text.split('\n') if line.strip()]

## 3. Feature Engineering for ML
We want to classify each line into: `HEADER`, `SUBJECT`, or `NOISE`.
We extract features like:
- Does it start with 'Semester'?
- Does it contain digits?
- Does it have 'Pass' or 'Fail'?
- Length of line.

In [ ]:
def extract_features(line):
    return {
        'starts_with_sem': 1 if line.lower().startswith('semester') else 0,
        'has_pass_fail': 1 if 'pass' in line.lower() or 'fail' in line.lower() else 0,
        'num_digits': sum(c.isdigit() for c in line),
        'length': len(line),
        'has_dash': 1 if '-' in line else 0,
        'word_count': len(line.split())
    }

X_data = [extract_features(line) for line in lines]
df_features = pd.DataFrame(X_data)
df_features['text'] = lines # Keep text for reference
print(df_features.head())

## 4. Train Model
We label a few samples manually to train a Random Forest Classifier.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Simple rule-based labeling for training set
def get_label(line):
    if re.search(r'Semester-\d+\s+.*(Pass|Fail|RA)', line, re.IGNORECASE):
        return 'SUBJECT'
    elif 'semester' in line.lower():
        return 'HEADER'
    else:
        return 'NOISE'

y_labels = [get_label(line) for line in lines]

clf = RandomForestClassifier(n_estimators=50)
clf.fit(df_features.drop('text', axis=1), y_labels)

print("Model Trained.")

## 5. Prediction & Parsing
Use the model to predict line types and extract data.

In [ ]:
predictions = clf.predict(df_features.drop('text', axis=1))

structured_data = []

for text, label in zip(lines, predictions):
    if label == 'SUBJECT':
        # If classified as subject, use Regex to extract details
        # The key change is capturing Grade Point (GP) before the Grade
        # Regex: Semester-X Code - Name GradePoint Grade Credits Result
        # Note the (\d+) before the grade group to capture GP (e.g. 7, 8, 9)
        match = re.search(r"Semester-(\d+)\s+(.+?)\s+(\d+)\s+([OABCPRF][+]?|RA|AB)\s+(\d+)\s+(Pass|Fail|RA)", text, re.IGNORECASE)
        if match:
            structured_data.append(match.groups())
            
df_results = pd.DataFrame(structured_data, columns=['Sem', 'Subject', 'GP', 'Grade', 'Credits', 'Result'])
print(df_results)